# Notebook 1 — Edge-case synthesis pipeline

Thin loop on a **real** photo:

`load → pick method per anomaly → edit → annotate → VLM judge → retry`

# Notebook 1 — Edge-case synthesis pipeline

Thin loop on a **real** photo:

`load → pick method per anomaly → edit → annotate → VLM judge → retry`

**Choose methods in [Notebook 1.5](01.5_method_comparison.ipynb)** (inpaint vs ControlNet vs instruct), then plug them in below as:

```python
METHOD_BY_ANOMALY = {
    "road_debris": "inpaint",
    "traffic_cone": "inpaint",
    "fog": "instruct",  # global effect — inpaint with a local hole usually fails
}
```

Pick `DATASET` + `HARDWARE` in the setup cell (defaults also in `configs/config.yaml`).

Domain: [Mapillary Vistas](https://www.mapillary.com/dataset/vistas) toy subset (CC BY-NC-SA). Also: `rdd2022`, `nordland_hf`.

| `HARDWARE` | Diffusion family | Judge |
|------------|------------------|-------|
| `cpu` | SD 1.5 inpaint + InstructPix2Pix | Qwen2.5-VL-3B |
| `gpu_l4` | FLUX.2-klein-4B (inpaint + instruct); SD 1.5 ControlNet | Qwen2.5-VL-7B |

---
## 0. Setup

```bash
cd implementations/edge_case_image_generation && uv sync
```

Select the project kernel, then run:

In [ ]:
import sys
from pathlib import Path


def _find_project_root() -> Path:
    here = Path.cwd().resolve()
    search = [here, *here.parents]
    for base in list(search):
        nested = base / "implementations" / "edge_case_image_generation"
        if nested.is_dir():
            search.append(nested)
    for base in search:
        if (base / "src" / "edgecase_synthesis").is_dir() and (base / "configs").is_dir():
            return base
    raise FileNotFoundError("Could not find edge_case_image_generation root")


PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("PROJECT_ROOT =", PROJECT_ROOT)

---
## 1. Load images + choose methods

Fill `METHOD_BY_ANOMALY` from Notebook 1.5. Valid methods: `inpaint`, `controlnet_dual`, `instruct`.

In [ ]:
import os

# Avoid flaky HF Xet downloads of large models ("Background writer channel closed").
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")

from edgecase_synthesis.compare_methods import COMPARE_METHODS, METHOD_SPECS
from edgecase_synthesis.config import load_config
from edgecase_synthesis.data import (
    ImageSample,
    get_data_source_info,
    list_sample_images,
    prepare_sample_images,
)
from edgecase_synthesis.pipeline import resolve_method_map
from edgecase_synthesis.viz import show_samples
from PIL import Image

HARDWARE = "cpu"  # or "gpu_l4"

# --- learner knobs (from Notebook 1.5) ---------------------------------
METHOD_BY_ANOMALY = {
    "pothole": "inpaint",
    "traffic_cone": "inpaint",
    "fog": "instruct",
}
MAX_RETRIES = 2  # judge decision == retry → re-edit with a new seed
# -----------------------------------------------------------------------

cfg = load_config(start=PROJECT_ROOT, overrides=[f"hardware={HARDWARE}"])
info = get_data_source_info(cfg)
prepare_sample_images(cfg=cfg)
samples_dir = Path(cfg.paths.samples_dir)

workshop = list(cfg.dataset.workshop_anomalies)
method_map = resolve_method_map(METHOD_BY_ANOMALY, workshop, cfg=cfg)

print(f"Hardware: {cfg.hardware.name}  family={cfg.generation.family}")
print(f"Source:   {info.label} ({info.license})")
print(f"Judge:    {cfg.judge.model_id}")
print("Method plan:")
for aid, method in method_map.items():
    print(f"  {aid:16s} → {method:16s}  ({METHOD_SPECS[method].title})")

# Prefer clean scene_* seeds (same idea as Notebook 1.5).
scene_paths = sorted(p for p in list_sample_images(samples_dir) if p.stem.startswith("scene_"))
if not scene_paths:
    raise RuntimeError(
        f"No scene_* images in {samples_dir}. Run scripts/extract_mapillary_toy.py"
    )
sample = ImageSample(
    path=scene_paths[0],
    image=Image.open(scene_paths[0]).convert("RGB"),
    name=scene_paths[0].stem,
)
print(f"Seed image: {sample.name}")
show_samples([sample], ncol=1, figsize=(8, 4));

---
## 2. Edit

Depth / segmentation run under the hood when a method needs them — you only pick the method.

In [ ]:
from edgecase_synthesis.compare_methods import MethodComparer
from edgecase_synthesis.conditioning import DepthEstimator, Segmenter
from edgecase_synthesis.pipeline import synthesize_one
from edgecase_synthesis.viz import save_generation_artifact, show_generation_result

depth_model = DepthEstimator.from_config(cfg)
segmenter = Segmenter.from_config(cfg)
comparer = MethodComparer.from_config(cfg)
print("depth:", depth_model.model_id)
print("seg:  ", segmenter.model_name)
print("edit: ", comparer.family, "on", comparer.device)

depth = depth_model.predict(sample.image)
seg = segmenter.predict(sample.image)

output_dir = Path(cfg.paths.outputs_dir)
results: dict[str, object] = {}

for anomaly_id, method in method_map.items():
    print("=" * 60)
    print(f"{anomaly_id}  via  {method}")
    syn = synthesize_one(
        sample.image,
        anomaly_id=anomaly_id,
        method=method,
        cfg=cfg,
        comparer=comparer,
        depth=depth,
        segmentation=seg,
        project_root=PROJECT_ROOT,
    )
    results[anomaly_id] = syn
    show_generation_result(sample, syn.generated)
    print(save_generation_artifact(sample, syn.generated, output_dir / "nb1" / anomaly_id))


---
## 3. Annotate

Open-vocab detector + SAM. Seed the edit mask when the method produced one (inpaint).

In [ ]:
from edgecase_synthesis.annotation import OpenVocabAnnotator
from edgecase_synthesis.config import load_anomaly
from edgecase_synthesis.viz import save_annotation_artifact, show_annotation_result

annotator = OpenVocabAnnotator.from_config(cfg)
base_classes = list(cfg.annotation.classes)
annotations = {}
dataset = str(cfg.dataset_name)

for anomaly_id, syn in results.items():
    anomaly_cfg = load_anomaly(dataset, anomaly_id, start=PROJECT_ROOT)
    anomaly_classes = list(anomaly_cfg.get("annotation_classes", []))
    classes = list(dict.fromkeys([*base_classes, *anomaly_classes]))
    seed_label = next((c for c in anomaly_classes if c not in {"road", "fog", "mist"}), None)
    annotation = annotator.annotate(
        syn.generated.image,
        classes=classes,
        seed_mask=syn.generated.edit_mask if seed_label else None,
        seed_label=seed_label,
    )
    annotations[anomaly_id] = annotation
    show_annotation_result(
        syn.generated.image,
        annotation,
        title=f"Annotations — {anomaly_id} ({syn.method})",
    )
    print(save_annotation_artifact(f"{sample.name}_{anomaly_id}", annotation, output_dir / "nb1"))

---
## 4. Judge + retry

Unload diffusion / detector first (GPU VRAM). On `retry`, re-edit with a new seed up to `MAX_RETRIES`.

In [ ]:
import gc

import torch
from edgecase_synthesis.judge import VLMJudge, summarize_annotations
from edgecase_synthesis.viz import save_judge_artifact, show_judge_result

for name in ("comparer", "annotator", "depth_model", "segmenter"):
    obj = globals().get(name)
    if obj is not None and hasattr(obj, "unload"):
        obj.unload()
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Reload editor stack only if we need retries (keep judge loaded otherwise).
judge = VLMJudge.from_config(cfg)
source_hint = str(cfg.dataset.get("source_hint", "a real photograph"))
print(judge.model_id, "threshold=", judge.threshold)

judgments = {}
final_results = dict(results)

for anomaly_id, syn in list(results.items()):
    anomaly_cfg = load_anomaly(dataset, anomaly_id, start=PROJECT_ROOT)
    attempt = 0
    current = syn
    while True:
        result = judge.judge(
            current.generated.image,
            prompt=current.generated.prompt,
            anomaly_id=anomaly_id,
            anomaly_name=str(anomaly_cfg.get("display_name", anomaly_id)),
            annotations_summary=summarize_annotations(annotations.get(anomaly_id)),
            source_hint=source_hint,
        )
        print(f"{anomaly_id} attempt={attempt} → {result.decision} ({result.overall:.1f})")
        show_judge_result(
            current.generated.image,
            result,
            title=f"Judge — {anomaly_id} [{current.method}] attempt {attempt}",
        )
        print(result.rationale)
        print(save_judge_artifact(f"{sample.name}_{anomaly_id}_a{attempt}", result, output_dir / "nb1"))

        if result.decision != "retry" or attempt >= MAX_RETRIES:
            judgments[anomaly_id] = result
            final_results[anomaly_id] = current
            break

        attempt += 1
        print(f"  retrying {anomaly_id} with seed_offset={attempt} …")
        # Reload pipelines for the retry edit.
        depth_model = DepthEstimator.from_config(cfg)
        segmenter = Segmenter.from_config(cfg)
        comparer = MethodComparer.from_config(cfg)
        depth = depth_model.predict(sample.image)
        seg = segmenter.predict(sample.image)
        current = synthesize_one(
            sample.image,
            anomaly_id=anomaly_id,
            method=method_map[anomaly_id],
            cfg=cfg,
            comparer=comparer,
            depth=depth,
            segmentation=seg,
            project_root=PROJECT_ROOT,
            seed_offset=attempt,
        )
        # Refresh annotation for the new image.
        annotator = OpenVocabAnnotator.from_config(cfg)
        anomaly_classes = list(anomaly_cfg.get("annotation_classes", []))
        classes = list(dict.fromkeys([*base_classes, *anomaly_classes]))
        seed_label = next((c for c in anomaly_classes if c not in {"road", "fog", "mist"}), None)
        annotations[anomaly_id] = annotator.annotate(
            current.generated.image,
            classes=classes,
            seed_mask=current.generated.edit_mask if seed_label else None,
            seed_label=seed_label,
        )
        comparer.unload()
        if hasattr(annotator, "unload"):
            annotator.unload()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

accepted = sum(1 for r in judgments.values() if r.decision == "accept")
print(f"Acceptance: {accepted}/{len(judgments)}")
for aid, r in judgments.items():
    print(f"  {aid:16s} {r.decision:7s}  method={method_map[aid]}")

---
## Wrap-up

| Notebook | Role |
|----------|------|
| **1.5** | Compare methods; decide `METHOD_BY_ANOMALY` |
| **1** (this) | Produce accepted samples with those choices |
| **2** | Batch over many seeds + retry policy |
| **3** | Train tiny detector: real vs real+accepted synthetic |

Add an anomaly: YAML under `configs/datasets/<dataset>/generation/anomalies/` + append id to `dataset.workshop_anomalies`, then pick a method for it in `METHOD_BY_ANOMALY`.